## CELL 1 : Memory-Optimized Loading and Dynamic Initialization

In [ ]:
import polars as pl
import torch
from torch_geometric.data import HeteroData
import networkx as nx
import matplotlib.pyplot as plt
import gc

PARQUET_PATH = "./processed/ml_ready_transactions.parquet"

# 1. Dynamically find 2 issuerCiks that ACTUALLY exist in the dataset
# We perform a fast, lazy scan to guarantee the CIKs exist without hardcoding.
print("Scanning dataset to dynamically select 2 companies...")
top_ciks = (
    pl.scan_parquet(PARQUET_PATH)
    .select("issuerCik")
    .drop_nulls()
    .group_by("issuerCik")
    .len()
    .sort("len", descending=True)
    .limit(2)
    .collect()
)
TARGET_CIKS = top_ciks["issuerCik"].to_list()
print(f"Selected issuerCiks: {TARGET_CIKS}")

# 2. Memory-efficient lazy query with robust date parsing
# We only project columns required by the schema and push predicates down to disk.
lazy_q = (
    pl.scan_parquet(PARQUET_PATH)
    .filter(pl.col("issuerCik").is_in(TARGET_CIKS))
    .drop_nulls(subset=["transactionDate", "rptOwnerCik", "securityTitle", "issuerCik"])
    .with_columns([
        # Parse transaction dates robustly from string format
        pl.col("transactionDate").str.to_datetime("%Y-%m-%d", strict=False).alias("transactionDate"),
        # Parse filing dates (BIGINT format like YYYYMMDD -> string -> datetime)
        pl.col("filingDate").cast(pl.Utf8).str.to_datetime("%Y%m%d", strict=False).alias("filingDate"),
        # Calculate derived transaction value and downcast to save RAM
        (pl.col("transactionShares") * pl.col("transactionPricePerShare")).cast(pl.Float32).alias("transactionValue")
    ])
    .sort("transactionDate")
)

df = lazy_q.collect()
print(f"Loaded {df.height} rows for the prototype graph.")
print(f"Estimated RAM: {df.estimated_size('mb'):.2f} MB")

## CELL 2: Leakage-Free Temporal Feature Engineering

In [ ]:
# 1. Sort strictly chronologically, then by Insider
# This is crucial so that shift() operations perfectly reflect historical sequences without leakage.
df = df.sort(["rptOwnerCik", "transactionDate"])

# 2. Map Trade Direction: Acquired (A) = 1, Disposed (D) = -1
df = df.with_columns([
    pl.when(pl.col("transactionAcquiredDisposedCode") == "A").then(1)
      .when(pl.col("transactionAcquiredDisposedCode") == "D").then(-1)
      .otherwise(0).alias("trade_direction")
])
df = df.with_columns([
    (pl.col("transactionShares") * pl.col("trade_direction")).alias("signed_shares")
])

# 3. Compute strictly historical (t-1) cumulative features
# By appending .shift(1) over the rptOwnerCik partition, feature vectors for transaction 't' 
# are constructed solely from data available at 't-1'.
df = df.with_columns([
    pl.col("transactionDate").shift(1).over("rptOwnerCik").alias("prev_transactionDate"),
    pl.col("transactionCode").cum_count().shift(1).over("rptOwnerCik").fill_null(0).alias("historical_trade_frequency"),
    (pl.col("transactionAcquiredDisposedCode") == "A").cast(pl.Int32).cum_sum().shift(1).over("rptOwnerCik").fill_null(0).alias("cum_buys"),
    (pl.col("transactionAcquiredDisposedCode") == "D").cast(pl.Int32).cum_sum().shift(1).over("rptOwnerCik").fill_null(0).alias("cum_sells"),
    pl.col("transactionValue").cum_sum().shift(1).over("rptOwnerCik").fill_null(0.0).alias("cum_value"),
    pl.col("signed_shares").cum_sum().shift(1).over("rptOwnerCik").fill_null(0.0).cast(pl.Float32).alias("ownership_change")
])

# 4. Derived historical ratios & deltas
df = df.with_columns([
    (pl.col("transactionDate") - pl.col("prev_transactionDate")).dt.total_days().cast(pl.Float32).fill_null(0.0).alias("days_since_previous_trade"),
    (pl.col("cum_buys") / pl.col("historical_trade_frequency")).fill_nan(0.0).fill_null(0.0).alias("historical_buy_ratio"),
    (pl.col("cum_sells") / pl.col("historical_trade_frequency")).fill_nan(0.0).fill_null(0.0).alias("historical_sell_ratio"),
    (pl.col("cum_value") / pl.col("historical_trade_frequency")).fill_nan(0.0).fill_null(0.0).alias("historical_average_trade_value"),
])

# 5. Calculate Rolling Window Features (30, 90, 180 days)
# Using group_by_dynamic with closed="left" ensures trades occurring on the current day are completely excluded from the window.
for days in [30, 90, 180]:
    rolling_df = df.group_by_dynamic(
        "transactionDate",
        by="rptOwnerCik",
        every="1d",
        period=f"{days}d",
        closed="left" 
    ).agg(pl.len().alias(f"trades_{days}d"))
    
    # Left join back to the main DataFrame on EXACT dates and insider ID
    df = df.join(rolling_df, on=["rptOwnerCik", "transactionDate"], how="left").with_columns(
        pl.col(f"trades_{days}d").fill_null(0)
    )

# Re-sort strictly by time for the Temporal Graph
df = df.sort("transactionDate")
display(df.select(["rptOwnerCik", "transactionDate", "days_since_previous_trade", "trades_30d", "historical_trade_frequency"]).head(3))

## CELL 3: PyTorch Geometric HeteroData Construction

In [ ]:
data = HeteroData()

# 1. Create Composite Security Key
# A 'Common Stock' issued by Microsoft is entirely different from a 'Common Stock' issued by Apple.
df = df.with_columns(
    (pl.col("issuerCik").cast(pl.Utf8) + "_" + pl.col("securityTitle")).alias("composite_security_id")
)

# 2. Extract Unique Entities & Create Continuous Index Mappings
unique_insiders = df["rptOwnerCik"].unique().to_list()
unique_companies = df["issuerCik"].unique().to_list()
unique_securities = df["composite_security_id"].unique().to_list()

insider_map = {idx: i for i, idx in enumerate(unique_insiders)}
company_map = {idx: i for i, idx in enumerate(unique_companies)}
security_map = {idx: i for i, idx in enumerate(unique_securities)}

data['Insider'].num_nodes = len(unique_insiders)
data['Company'].num_nodes = len(unique_companies)
data['Security'].num_nodes = len(unique_securities)

# 3. Build Static Edge: Insider -> associated_with -> Company
# Keeps distinct role relationships as they change over time rather than arbitrary `.unique(keep="first")`
df_assoc = df.select(["rptOwnerCik", "issuerCik", "isDirector", "isOfficer", "isTenPercentOwner"]).unique()
src_assoc = [insider_map[x] for x in df_assoc["rptOwnerCik"].to_list()]
dst_assoc = [company_map[x] for x in df_assoc["issuerCik"].to_list()]

data['Insider', 'associated_with', 'Company'].edge_index = torch.tensor([src_assoc, dst_assoc], dtype=torch.long)
data['Insider', 'associated_with', 'Company'].edge_attr = torch.tensor(
    df_assoc.select(["isDirector", "isOfficer", "isTenPercentOwner"]).fill_null(0).to_numpy(), 
    dtype=torch.float32
)

# 4. Build Static Edge: Company -> issues -> Security
df_issues = df.select(["issuerCik", "composite_security_id"]).unique()
src_issues = [company_map[x] for x in df_issues["issuerCik"].to_list()]
dst_issues = [security_map[x] for x in df_issues["composite_security_id"].to_list()]
data['Company', 'issues', 'Security'].edge_index = torch.tensor([src_issues, dst_issues], dtype=torch.long)

# 5. Build Temporal Edge: Insider -> trades -> Security
# PRESERVES EVERY EVENT (Duplicates are not collapsed)
src_trades = [insider_map[x] for x in df["rptOwnerCik"].to_list()]
dst_trades = [security_map[x] for x in df["composite_security_id"].to_list()]
timestamps = df["transactionDate"].dt.timestamp("ms").fill_null(0).to_list()

data['Insider', 'trades', 'Security'].edge_index = torch.tensor([src_trades, dst_trades], dtype=torch.long)
data['Insider', 'trades', 'Security'].time = torch.tensor(timestamps, dtype=torch.long)

# Encode categorical variables for the tensor
df = df.with_columns([
    pl.col("transactionCode").cast(pl.Categorical).to_physical().cast(pl.Float32).alias("transactionCode_encoded"),
    pl.col("directOrIndirectOwnership").cast(pl.Categorical).to_physical().cast(pl.Float32).alias("ownership_encoded"),
    pl.col("filingDate").dt.timestamp("ms").cast(pl.Float32).alias("filing_timestamp")
])

trade_feature_cols = [
    "transactionCode_encoded", "transactionShares", "transactionPricePerShare", "transactionValue",
    "ownership_encoded", "filing_timestamp", "days_since_previous_trade", "trades_30d", "trades_90d", 
    "trades_180d", "historical_trade_frequency", "historical_buy_ratio", "historical_sell_ratio", 
    "historical_average_trade_value", "ownership_change"
]

data['Insider', 'trades', 'Security'].edge_attr = torch.tensor(
    df.select(trade_feature_cols).fill_null(0.0).to_numpy(),
    dtype=torch.float32
)

# Clean up memory
del df_assoc, df_issues
gc.collect()

# Print clear summary
print(f"\n--- Graph Summary ---")
print(f"Nodes: {data.num_nodes} (Insiders: {len(unique_insiders)}, Companies: {len(unique_companies)}, Securities: {len(unique_securities)})")
print(f"Edges: {data.num_edges} (Trades: {len(src_trades)}, Associations: {len(src_assoc)}, Issues: {len(src_issues)})")
print(f"Trade Feature Dimension: {data['Insider', 'trades', 'Security'].edge_attr.shape[1]}")

## CELL 4: Clean Structural Visualization & Timeline

In [ ]:
# 1. Structural Visualization (Aggregates edges to prevent overlapping spaghetti)
G_struct = nx.DiGraph()
vis_limit = 10 # Cap visual size

for i in range(min(vis_limit, len(src_assoc))):
    insider = f"I_{src_assoc[i]}"
    company = f"C_{dst_assoc[i]}"
    G_struct.add_node(insider, type='Insider', color='lightblue')
    G_struct.add_node(company, type='Company', color='lightgreen')
    G_struct.add_edge(insider, company, label='associated')

for i in range(min(vis_limit, len(src_issues))):
    company = f"C_{src_issues[i]}"
    security = f"S_{dst_issues[i]}"
    G_struct.add_node(company, type='Company', color='lightgreen')
    G_struct.add_node(security, type='Security', color='salmon')
    G_struct.add_edge(company, security, label='issues')

# Aggregate trades purely for structural visualization
trade_counts = df.group_by(["rptOwnerCik", "composite_security_id"]).len()
for row in trade_counts.head(vis_limit).iter_rows(named=True):
    insider = f"I_{insider_map[row['rptOwnerCik']]}"
    sec = f"S_{security_map[row['composite_security_id']]}"
    if G_struct.has_node(insider) and G_struct.has_node(sec):
        G_struct.add_edge(insider, sec, label=f"trades ({row['len']}x)")

plt.figure(figsize=(10, 6))
pos = nx.spring_layout(G_struct, seed=42)
node_colors = [nx.get_node_attributes(G_struct, 'color').get(n, 'gray') for n in G_struct.nodes()]

nx.draw(G_struct, pos, with_labels=True, node_color=node_colors, node_size=1500, font_size=8, font_weight='bold', edge_color='gray')
nx.draw_networkx_edge_labels(G_struct, pos, edge_labels=nx.get_edge_attributes(G_struct, 'label'), font_size=7)
plt.title("Aggregated Graph Structure (Prototype)")
plt.show()

# 2. Temporal Timeline Visualization (Tracking distinct temporal events)
plt.figure(figsize=(12, 4))
timeline_df = df.select(["transactionDate", "transactionValue"]).drop_nulls().sort("transactionDate")
if timeline_df.height > 1000:
    timeline_df = timeline_df.sample(1000).sort("transactionDate") # Sample to prevent plotting freeze
    
plt.scatter(timeline_df["transactionDate"].to_list(), timeline_df["transactionValue"].to_list(), alpha=0.5, s=15, c='purple', edgecolors='none')
plt.title(f"Temporal Trade Events Timeline (n={timeline_df.height})")
plt.xlabel("Transaction Date")
plt.ylabel("Transaction Value ($)")
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.show()

# --- Scalability Explanation ---
print("\n--- Scaling to the Full Dataset ---")
print("1. Out-of-Core Processing: We will replace df.collect() with chunked iteration or Dask, mapping entities dynamically without maintaining a massive mapping dictionary in RAM.")
print("2. Memory Management: Trade nodes will not be retained in pandas/polars formats. The processed tensors will be written natively to disk-backed formats (e.g., PyG's on-disk dataset format).")
print("3. Temporal Batches: For training, PyTorch Geometric's TemporalDataLoader dynamically constructs chronological neighbor subgraphs in minibatches. The GPU will only ever 'see' a few thousand edges at a time, allowing continuous ingestion of the 600M edge dataset.")